In [ ]:
!pip install triton

In [ ]:
# ───────────────────────── imports ─────────────────────────
import os, json, time
import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


# ───────────────────────── imports ─────────────────────────
import os, json, time
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F    
from torch.utils.data import DataLoader, Dataset
import triton
import triton.language as tl
import torchvision.transforms as T

# ──────────────────── EuroSAT CSV Dataset ──────────────────
class EuroSATDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        with open(os.path.join(root_dir, 'label_map.json'), 'r') as f:
            self.label_map = json.load(f)                   # {"AnnualCrop":0,…}
        self.num_to_idx = {str(v): v for v in self.label_map.values()}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.root_dir, row['Filename'])).convert('RGB')
        label = self.num_to_idx[str(row['Label'])]
        if self.transform: img = self.transform(img)
        return img, label



dim = 128          # Embedding dimension (input & output channels)
hidden = dim * 2  # Hidden dimension (often 2× for MLPs)


class AGeLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Learnable parameters per channel (or scalar if dim=1)
        self.alpha = nn.Parameter(torch.ones(dim))
        self.beta  = nn.Parameter(torch.ones(dim))
        self.gamma = nn.Parameter(torch.zeros(dim))
        self.theta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # x: shape (..., dim)
        return self.beta * F.gelu(self.alpha * x + self.gamma) + self.theta

class IMLP(nn.Module):
    """
    Improved MLP (IMLP) with multiple AGeLU activations and optional spatial-wise enhancement via depthwise conv.
    """
    def __init__(self, dim, hidden, spatial_ch=None):
        """
        Args:
            dim (int): input & output feature dimension.
            hidden (int): hidden (intermediate) dimension.
            spatial_ch (int or None): If set, applies a depthwise 3x3 conv + channel projection after channel MLP.
        """
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden, bias=False)
        self.agelu1 = AGeLU(hidden)
        # Optionally, you could stack multiple AGeLU layers:
        self.agelu2 = AGeLU(hidden)
        self.fc2 = nn.Linear(hidden, dim, bias=False)
        self.spatial = None
        if spatial_ch is not None:
            # Spatial enhancement: Depthwise 3×3 conv preserving dim
            self.spatial = nn.Sequential(
                nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=False),
                nn.BatchNorm2d(dim),
                nn.GELU()
            )
        self.norm = nn.Identity()  # placeholder; optionally add BatchNorm or LayerNorm

    def forward(self, x):
        """
        x: tensor with shape (B, C, H, W) or (B, N, C)
        We'll apply fc across the channel dimension while preserving shape.
        """
        orig_shape = x.shape
        if x.dim() == 4:
            B, C, H, W = x.shape
            x_flat = x.flatten(2).transpose(1, 2)  # B, H*W, C
            y = self.fc1(x_flat)
            y = self.agelu1(y)
            y = self.agelu2(y)
            y = self.fc2(y)
            y = self.norm(y)
            y = y.transpose(1, 2).view(B, C, H, W)
        elif x.dim() == 3:
            B, N, C = x.shape
            y = self.fc1(x)
            y = self.agelu1(y)
            y = self.agelu2(y)
            y = self.fc2(y)
            y = self.norm(y)
        else:
            raise ValueError(f"Unsupported input dimensions: {x.shape}")

        if self.spatial is not None and x.dim() == 4:
            y = y + self.spatial(y)

        return y
        
mlp = IMLP(dim, hidden, spatial_ch=dim)


# Helper functions
@torch.no_grad()
def _wpart(x, w):
    B, C, H, W = x.shape
    return x.view(B, C, H//w, w, W//w, w).permute(0, 2, 4, 1, 3, 5).reshape(-1, C, w, w)

@torch.no_grad()
def _wunpart(x, w, H, W, B):
    nh, nw = H//w, W//w
    return x.view(B, nh, nw, -1, w, w).permute(0, 3, 1, 4, 2, 5).reshape(B, -1, H, W)

# Triton kernels for Flash Window Attention
@triton.jit
def _flash_window_fwd_kernel(
    Q, K, V, O,
    scale_qk: tl.constexpr,
    batch: tl.constexpr,
    head: tl.constexpr,
    seq: tl.constexpr,
    head_dim: tl.constexpr,
    head_chunk: tl.constexpr,
    chunk_dim: tl.constexpr,
):
    batch_id = tl.program_id(0)
    head_id = tl.program_id(1)

    stride_head = seq * head_dim
    stride_batch = stride_head * head
    offset = batch_id * stride_batch + head_id * stride_head

    # Initialize attention matrix in SRAM
    attn = tl.zeros((seq, seq), dtype=tl.float32)

    # Load Q, K chunks
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        attn += tl.dot(q_data, k_data.trans(1, 0))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))

    # Scale and softmax
    attn *= scale_qk
    attn -= tl.max(attn, axis=1, keep_dims=True)
    attn = tl.math.exp(attn)
    p_sum = tl.sum(attn, axis=1, keep_dims=True)
    attn /= p_sum
    attn = attn.to(Q.dtype.element_ty)

    # Compute output
    V_ptr = tl.make_block_ptr(
        base=V + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    O_ptr = tl.make_block_ptr(
        base=O + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        v_data = tl.load(V_ptr, boundary_check=(0, 1), padding_option="zero")
        o_data = tl.dot(attn, v_data)
        tl.store(O_ptr, o_data.to(Q.dtype.element_ty), boundary_check=(0, 1))
        V_ptr = tl.advance(V_ptr, (0, chunk_dim))
        O_ptr = tl.advance(O_ptr, (0, chunk_dim))

@triton.jit
def _flash_window_bwd_kernel(
    Q, K, V, dO,
    dQ, dK, dV,
    scale_qk: tl.constexpr,
    batch: tl.constexpr,
    head: tl.constexpr,
    seq: tl.constexpr,
    head_dim: tl.constexpr,
    head_chunk: tl.constexpr,
    chunk_dim: tl.constexpr,
):
    batch_id = tl.program_id(0)
    head_id = tl.program_id(1)

    stride_head = seq * head_dim
    stride_batch = stride_head * head
    offset = batch_id * stride_batch + head_id * stride_head

    # Compute attention matrix
    attn = tl.zeros((seq, seq), dtype=tl.float32)
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        attn += tl.dot(q_data, k_data.trans(1, 0))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))

    # Scale and softmax
    attn *= scale_qk
    attn -= tl.max(attn, axis=1, keep_dims=True)
    attn = tl.math.exp(attn)
    p_sum = tl.sum(attn, axis=1, keep_dims=True)
    attn /= p_sum
    attn = attn.to(Q.dtype.element_ty)

    # Compute dV and dP
    dP = tl.zeros((seq, seq), dtype=tl.float32)
    V_ptr = tl.make_block_ptr(
        base=V + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dO_ptr = tl.make_block_ptr(
        base=dO + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dV_ptr = tl.make_block_ptr(
        base=dV + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        v_data = tl.load(V_ptr, boundary_check=(0, 1), padding_option="zero")
        do_data = tl.load(dO_ptr, boundary_check=(0, 1), padding_option="zero")
        dV_data = tl.dot(attn.trans(1, 0), do_data)
        tl.store(dV_ptr, dV_data.to(V.dtype.element_ty), boundary_check=(0, 1))
        dP += tl.dot(do_data, v_data.trans(1, 0))
        V_ptr = tl.advance(V_ptr, (0, chunk_dim))
        dO_ptr = tl.advance(dO_ptr, (0, chunk_dim))
        dV_ptr = tl.advance(dV_ptr, (0, chunk_dim))

    # Compute dS
    dS = attn * (dP - tl.sum(attn * dP, axis=1, keep_dims=True))

    # Compute dQ, dK
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dQ_ptr = tl.make_block_ptr(
        base=dQ + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dK_ptr = tl.make_block_ptr(
        base=dK + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        dQ_data = tl.dot(dS, k_data)
        tl.store(dQ_ptr, dQ_data.to(Q.dtype.element_ty), boundary_check=(0, 1))
        dK_data = tl.dot(dS.trans(1, 0), q_data)
        tl.store(dK_ptr, dK_data.to(K.dtype.element_ty), boundary_check=(0, 1))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))
        dQ_ptr = tl.advance(dQ_ptr, (0, chunk_dim))
        dK_ptr = tl.advance(dK_ptr, (0, chunk_dim))

# Autograd function for Flash Window Attention
class FlashWindowAttnFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, scale_qk):
        batch, heads, seq, head_dim = q.shape
        assert head_dim % 16 == 0, "head_dim must be divisible by 16"
        o = torch.empty_like(q)
        head_chunk = head_dim // 16  # r = C/16 as per paper
        grid = (batch, heads, 1)
        _flash_window_fwd_kernel[grid](
            q, k, v, o, scale_qk,
            batch, heads, seq, head_dim, head_chunk, 16
        )
        ctx.save_for_backward(q, k, v)
        ctx.scale_qk = scale_qk
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v = ctx.saved_tensors
        batch, heads, seq, head_dim = q.shape
        dq = torch.empty_like(q)
        dk = torch.empty_like(k)
        dv = torch.empty_like(v)
        head_chunk = head_dim // 16
        grid = (batch, heads, 1)
        _flash_window_bwd_kernel[grid](
            q, k, v, do, dq, dk, dv,
            ctx.scale_qk, batch, heads, seq, head_dim, head_chunk, 16
        )
        return dq, dk, dv, None

# Flash Window Multi-Head Self-Attention
# Flash Window Multi-Head Self-Attention
class FlashWindowMSA(nn.Module):
    def __init__(self, dim, heads, win, qk_dim=None):
        super().__init__()
        self.w = win
        self.heads = heads
        self.qk_dim = qk_dim if qk_dim is not None else dim

        self.qkv = nn.Linear(dim, dim * 3)
        head_dim = dim // heads
        qk_head_dim = self.qk_dim // heads  # expanded head dim when qk_dim is provided

        # project Q and K to qk_head_dim; ALSO project V to the same dim
        self.qk_proj = nn.Linear(head_dim, qk_head_dim) if qk_dim is not None else nn.Identity()
        self.v_proj  = nn.Linear(head_dim, qk_head_dim) if qk_dim is not None else nn.Identity()

        self.scale = (self.qk_dim // heads) ** -0.5  # scale matches projected QK dim

        # output projection must accept heads * qk_head_dim (not dim) when qk_dim is used
        self.proj = nn.Linear(qk_head_dim * heads, dim)

    def forward(self, x):  # x: B C H W
        
        B, C, H, W = x.shape
        w = self.w
    
        # Dynamic padding (so H, W are divisible by w)
        pad_h = (w - H % w) % w
        pad_w = (w - W % w) % w
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        H_pad, W_pad = H + pad_h, W + pad_w
    
        # Window partition → tokens (B_win, L, C)
        tokens = _wpart(x, w).flatten(2).transpose(1, 2)
        B_win, L, _ = tokens.shape
        qkv = self.qkv(tokens).reshape(B_win, L, 3, self.heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B*nh*nw) heads L head_dim

        # project all of Q, K, and V to the same head dim
        q = self.qk_proj(q)
        k = self.qk_proj(k)
        v = self.v_proj(v)

        out = FlashWindowAttnFunc.apply(q, k, v, self.scale)

        # Project and unpartition
        out = self.proj(out.transpose(1, 3).reshape(B_win, L, -1))  # (B*nh*nw) L C
        out = out.transpose(1, 2).view(-1, C, w, w)
        out = _wunpart(out, w, H_pad, W_pad, B)

        # Remove padding
        if pad_h or pad_w:
            out = out[:, :, :H, :W]
        return out

# Depth-wise Separable Convolution Block
class DWConvBN(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch), nn.GELU(),
            nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x):
        return self.net(x)


# Fractal Information-Flow Attention (L1 + L2)
class FIFMAtt(nn.Module):
    def __init__(self, dim, heads, p=4, s=2, tau=1.0):
        super().__init__()
        self.tau = tau
        self.p = p      # Local window size
        self.s = s      # Grouping factor  
        self.P = p * s  # Non-local window size
        
        # Layer norms for L1 and L2 levels
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        
        # L1: Local fractal layer (p×p windows)
        assert (dim // heads) % 16 == 0, f"head_dim ({dim // heads}) must be divisible by 16"
        self.l1 = FlashWindowMSA(dim, heads, p)
        
        # L2: Non-local fractal aggregation (P×P windows with higher-dim QK space)
        self.l2 = FlashWindowMSA(dim, heads, self.P, qk_dim=dim*2)
        
    def _ln_chw(self, x, ln):
        """Apply LayerNorm to CHW format tensor"""
        B, C, H, W = x.shape
        y = x.flatten(2).transpose(1, 2)
        y = ln(y).transpose(1, 2).view(B, C, H, W)
        return y
    
    def _fractal_reshape_l2(self, x):
        """Group s×s local patches into P×P regions for L2"""
        B, C, H, W = x.shape
        p, s, P = self.p, self.s, self.P
        
        # Dynamic padding
        pad_h = (P - H % P) % P
        pad_w = (P - W % P) % P 
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
            H_pad, W_pad = H + pad_h, W + pad_w
        else:
            H_pad, W_pad = H, W
            
        # Fractal aggregation: p×p → P×P grouping
        nh_p, nw_p = H_pad // p, W_pad // p
        x_local = x.view(B, C, nh_p, p, nw_p, p)
        
        nh_P, nw_P = nh_p // s, nw_p // s
        x_grouped = x_local.view(B, C, nh_P, s, p, nw_P, s, p)
        x_fractal = x_grouped.permute(0, 2, 5, 1, 3, 6, 4, 7)
        x_fractal = x_fractal.reshape(B * nh_P * nw_P, C, P, P)
        
        return x_fractal, (H_pad, W_pad, nh_P, nw_P, pad_h, pad_w)
    
    def _fractal_unreshape_l2(self, x, reshape_info):
        """Reverse fractal aggregation"""
        H_pad, W_pad, nh_P, nw_P, pad_h, pad_w = reshape_info
        B = x.shape[0] // (nh_P * nw_P)
        C, P = x.shape[1], x.shape[2]
        p, s = self.p, self.s
        
        x = x.view(B, nh_P, nw_P, C, s, s, p, p)
        x = x.permute(0, 3, 1, 4, 6, 2, 5, 7)
        x = x.reshape(B, C, nh_P * s * p, nw_P * s * p)
        
        if pad_h or pad_w:
            x = x[:, :, :H_pad-pad_h, :W_pad-pad_w]
            
        return x
    
    def forward(self, x):
        """L1 (local) → L2 (non-local fractal aggregation)"""
        # L1: Local Fractal Layer
        x_ln1 = self._ln_chw(x, self.ln1)
        y1 = self.l1(x_ln1)
        x = x + self.tau * y1
        
        # L2: Non-Local Fractal Aggregation Layer  
        x_fractal, reshape_info = self._fractal_reshape_l2(x)
        y2 = self.l2(x_fractal)
        y2_reshaped = self._fractal_unreshape_l2(y2, reshape_info)
        y2_ln = self._ln_chw(y2_reshaped, self.ln2)
        x = x + self.tau * y2_ln
        
        return x

# Fractal Feed-Forward Convolution (L3)
class FIFMConv(nn.Module):
    """L3 Global Modelling Layer using your existing IMLP"""
    def __init__(self, dim, mlp_ratio=2.0, tau=1.0):
        super().__init__()
        self.tau = tau
        self.ln = nn.LayerNorm(dim)
        
        # Spatial convolution for global context
        self.dw_conv = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim, bias=False),
            nn.BatchNorm2d(dim),
            nn.GELU()
        )
        
        # Use your existing IMLP
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = IMLP(dim, hidden_dim, spatial_ch=None)
        
    def _ln_chw(self, x):
        B, C, H, W = x.shape
        y = x.flatten(2).transpose(1, 2)
        y = self.ln(y).transpose(1, 2).view(B, C, H, W)
        return y
        
    def forward(self, x):
        x_ln = self._ln_chw(x)
        y = self.dw_conv(x_ln)  # Spatial processing
        y = self.mlp(y)         # Your IMLP for channel processing
        return x + self.tau * y
# Fractal Block
class FracBlock(nn.Module):
    def __init__(self, dim, heads, p, s, tau=1.0, mlp_ratio=2.0):
        super().__init__()
        self.att = FIFMAtt(dim, heads, p, s, tau)
        self.conv = FIFMConv(dim, mlp_ratio=mlp_ratio, tau=tau)
    def forward(self, x):
        x = self.att(x)
        x = self.conv(x)
        return x

# Fractal Classification Model
class FractalCLS(nn.Module):
    CFG = {
        'T': dict(ed=64, depths=[2, 2, 6, 2], heads=[2, 4, 8, 16]),
        'S': dict(ed=48, depths=[1, 2, 4, 2], heads=[3, 6, 12, 24]),
        'B': dict(ed=128, depths=[2, 2, 18, 2], heads=[4, 8, 16, 32]),
    }

    def __init__(self, n_cls=10, size='T', p=4, s=2, tau=1.0):
        super().__init__()
        cfg = self.CFG[size]
        self.stem = nn.Sequential(
            DWConvBN(3, cfg['ed'], stride=p),
            nn.BatchNorm2d(cfg['ed']), nn.GELU())
        dims = [cfg['ed'], cfg['ed']*2, cfg['ed']*4, cfg['ed']*8]
        wins = [p, p, p, p]  # Fix: Use p=4 for all stages to avoid dimension reduction
        stages = []
        for i in range(4):
            stages += [FracBlock(dims[i], cfg['heads'][i], wins[i], s, tau)
                       for _ in range(cfg['depths'][i])]
            if i < 3:
                stages.append(nn.Sequential(
                    DWConvBN(dims[i], dims[i+1], stride=2),
                    nn.BatchNorm2d(dims[i+1]), nn.GELU()))
        self.net = nn.Sequential(*stages)
        self.norm = nn.BatchNorm2d(dims[-1])
        self.head = nn.Linear(dims[-1], n_cls)

    def forward(self, x):
        x = self.stem(x)
        x = self.net(x)
        x = self.norm(x)
        return self.head(torch.mean(x, dim=[2, 3]))


# ──────────────── training / evaluation util ───────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item(); total += y.size(0)
    return 100. * correct / total

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); running_loss = 0.
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y); loss.backward(); optimizer.step()
        running_loss += loss.item() * x.size(0)
    return running_loss / len(loader.dataset)


# ─────────────────────────── main ──────────────────────────
def main():
    # ---------- paths ---------------------------------------------------------
    ROOT   = "/kaggle/input/eurosat-dataset/EuroSAT"
    CSV_T  = f"{ROOT}/train.csv"
    CSV_V  = f"{ROOT}/validation.csv"
    CSV_TE = f"{ROOT}/test.csv"

    # ---------- hyper-params ---------------------------------------------------
    BATCH_SIZE   = 32
    EPOCHS_FRAC  = 15          # tweak as needed
    NUM_CLASSES  = 10
    LEARNING_RATE= 6e-3

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"▶ Device: {device}")

    tfm = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225])
    ])

    # ---------- data loaders ---------------------------------------------------
    train_loader = DataLoader(EuroSATDataset(CSV_T, ROOT, tfm), batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(EuroSATDataset(CSV_V, ROOT, tfm), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(EuroSATDataset(CSV_TE, ROOT, tfm), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    # ====================  FractalCLS train ===============================
    print("\n🧩 Training FractalCLS-Small (size='S')")
    fractal = FractalCLS(NUM_CLASSES, size='S').to(device)

    opt   = optim.AdamW(fractal.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FRAC)
    crit  = nn.CrossEntropyLoss()

    for epoch in range(1, EPOCHS_FRAC + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(fractal, train_loader, crit, opt, device)
        val_acc = evaluate(fractal, val_loader, device)
        sched.step()
        print(f"  Epoch {epoch:02d}/{EPOCHS_FRAC} | "
              f"loss {tr_loss:.4f} | val acc {val_acc:5.2f}% | "
              f"{(time.time()-t0):.1f}s | lr {sched.get_last_lr()[0]:.2e}")

    test_acc_frac = evaluate(fractal, test_loader, device)
    print(f"\n📈 FractalCLS Test Accuracy: {test_acc_frac:.2f}%")
    # --- Confusion Matrix ---
    try:
        with open(os.path.join(ROOT, "label_map.json"), "r") as f:
            _lm = json.load(f)  # e.g., {"AnnualCrop": 0, "Forest": 1, ...}
        # Sort so labels follow numeric order 0..NUM_CLASSES-1
        class_names = [k for k, v in sorted(_lm.items(), key=lambda kv: kv[1])]
    except Exception:
        class_names = list(range(NUM_CLASSES))
    
    y_true, y_pred = [], []
    fractal.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = fractal(x).argmax(1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Display + Save
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45, ax=ax, colorbar=True)
    plt.title("Confusion Matrix - FractalCLS on EuroSAT")
    
    # Save to file
    plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
    
    # Show on screen
    plt.show()
    
    print("✅ Confusion matrix saved as 'confusion_matrix.png'")
        
    # 🔐 Save the trained model weights
    torch.save(fractal.state_dict(), "fractalcls_s_eurosat.pth")
    print("✅ Model checkpoint saved as 'fractalcls_s_eurosat.pth'")


if __name__ == "__main__":
    main()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl
import torch
import torch.nn as nn
import torch.nn.functional as F


dim = 128          # Embedding dimension (input & output channels)
hidden = dim * 2  # Hidden dimension (often 2× for MLPs)


class AGeLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Learnable parameters per channel (or scalar if dim=1)
        self.alpha = nn.Parameter(torch.ones(dim))
        self.beta  = nn.Parameter(torch.ones(dim))
        self.gamma = nn.Parameter(torch.zeros(dim))
        self.theta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # x: shape (..., dim)
        return self.beta * F.gelu(self.alpha * x + self.gamma) + self.theta

class IMLP(nn.Module):
    """
    Improved MLP (IMLP) with multiple AGeLU activations and optional spatial-wise enhancement via depthwise conv.
    """
    def __init__(self, dim, hidden, spatial_ch=None):
        """
        Args:
            dim (int): input & output feature dimension.
            hidden (int): hidden (intermediate) dimension.
            spatial_ch (int or None): If set, applies a depthwise 3x3 conv + channel projection after channel MLP.
        """
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden, bias=False)
        self.agelu1 = AGeLU(hidden)
        # Optionally, you could stack multiple AGeLU layers:
        self.agelu2 = AGeLU(hidden)
        self.fc2 = nn.Linear(hidden, dim, bias=False)
        self.spatial = None
        if spatial_ch is not None:
            # Spatial enhancement: Depthwise 3×3 conv preserving dim
            self.spatial = nn.Sequential(
                nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=False),
                nn.BatchNorm2d(dim),
                nn.GELU()
            )
        self.norm = nn.Identity()  # placeholder; optionally add BatchNorm or LayerNorm

    def forward(self, x):
        """
        x: tensor with shape (B, C, H, W) or (B, N, C)
        We'll apply fc across the channel dimension while preserving shape.
        """
        orig_shape = x.shape
        if x.dim() == 4:
            B, C, H, W = x.shape
            x_flat = x.flatten(2).transpose(1, 2)  # B, H*W, C
            y = self.fc1(x_flat)
            y = self.agelu1(y)
            y = self.agelu2(y)
            y = self.fc2(y)
            y = self.norm(y)
            y = y.transpose(1, 2).view(B, C, H, W)
        elif x.dim() == 3:
            B, N, C = x.shape
            y = self.fc1(x)
            y = self.agelu1(y)
            y = self.agelu2(y)
            y = self.fc2(y)
            y = self.norm(y)
        else:
            raise ValueError(f"Unsupported input dimensions: {x.shape}")

        if self.spatial is not None and x.dim() == 4:
            y = y + self.spatial(y)

        return y
        
mlp = IMLP(dim, hidden, spatial_ch=dim)


# Helper functions
@torch.no_grad()
def _wpart(x, w):
    B, C, H, W = x.shape
    return x.view(B, C, H//w, w, W//w, w).permute(0, 2, 4, 1, 3, 5).reshape(-1, C, w, w)

@torch.no_grad()
def _wunpart(x, w, H, W, B):
    nh, nw = H//w, W//w
    return x.view(B, nh, nw, -1, w, w).permute(0, 3, 1, 4, 2, 5).reshape(B, -1, H, W)

# Triton kernels for Flash Window Attention
@triton.jit
def _flash_window_fwd_kernel(
    Q, K, V, O,
    scale_qk: tl.constexpr,
    batch: tl.constexpr,
    head: tl.constexpr,
    seq: tl.constexpr,
    head_dim: tl.constexpr,
    head_chunk: tl.constexpr,
    chunk_dim: tl.constexpr,
):
    batch_id = tl.program_id(0)
    head_id = tl.program_id(1)

    stride_head = seq * head_dim
    stride_batch = stride_head * head
    offset = batch_id * stride_batch + head_id * stride_head

    # Initialize attention matrix in SRAM
    attn = tl.zeros((seq, seq), dtype=tl.float32)

    # Load Q, K chunks
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        attn += tl.dot(q_data, k_data.trans(1, 0))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))

    # Scale and softmax
    attn *= scale_qk
    attn -= tl.max(attn, axis=1, keep_dims=True)
    attn = tl.math.exp(attn)
    p_sum = tl.sum(attn, axis=1, keep_dims=True)
    attn /= p_sum
    attn = attn.to(Q.dtype.element_ty)

    # Compute output
    V_ptr = tl.make_block_ptr(
        base=V + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    O_ptr = tl.make_block_ptr(
        base=O + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        v_data = tl.load(V_ptr, boundary_check=(0, 1), padding_option="zero")
        o_data = tl.dot(attn, v_data)
        tl.store(O_ptr, o_data.to(Q.dtype.element_ty), boundary_check=(0, 1))
        V_ptr = tl.advance(V_ptr, (0, chunk_dim))
        O_ptr = tl.advance(O_ptr, (0, chunk_dim))

@triton.jit
def _flash_window_bwd_kernel(
    Q, K, V, dO,
    dQ, dK, dV,
    scale_qk: tl.constexpr,
    batch: tl.constexpr,
    head: tl.constexpr,
    seq: tl.constexpr,
    head_dim: tl.constexpr,
    head_chunk: tl.constexpr,
    chunk_dim: tl.constexpr,
):
    batch_id = tl.program_id(0)
    head_id = tl.program_id(1)

    stride_head = seq * head_dim
    stride_batch = stride_head * head
    offset = batch_id * stride_batch + head_id * stride_head

    # Compute attention matrix
    attn = tl.zeros((seq, seq), dtype=tl.float32)
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        attn += tl.dot(q_data, k_data.trans(1, 0))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))

    # Scale and softmax
    attn *= scale_qk
    attn -= tl.max(attn, axis=1, keep_dims=True)
    attn = tl.math.exp(attn)
    p_sum = tl.sum(attn, axis=1, keep_dims=True)
    attn /= p_sum
    attn = attn.to(Q.dtype.element_ty)

    # Compute dV and dP
    dP = tl.zeros((seq, seq), dtype=tl.float32)
    V_ptr = tl.make_block_ptr(
        base=V + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dO_ptr = tl.make_block_ptr(
        base=dO + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dV_ptr = tl.make_block_ptr(
        base=dV + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        v_data = tl.load(V_ptr, boundary_check=(0, 1), padding_option="zero")
        do_data = tl.load(dO_ptr, boundary_check=(0, 1), padding_option="zero")
        dV_data = tl.dot(attn.trans(1, 0), do_data)
        tl.store(dV_ptr, dV_data.to(V.dtype.element_ty), boundary_check=(0, 1))
        dP += tl.dot(do_data, v_data.trans(1, 0))
        V_ptr = tl.advance(V_ptr, (0, chunk_dim))
        dO_ptr = tl.advance(dO_ptr, (0, chunk_dim))
        dV_ptr = tl.advance(dV_ptr, (0, chunk_dim))

    # Compute dS
    dS = attn * (dP - tl.sum(attn * dP, axis=1, keep_dims=True))

    # Compute dQ, dK
    Q_ptr = tl.make_block_ptr(
        base=Q + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    K_ptr = tl.make_block_ptr(
        base=K + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dQ_ptr = tl.make_block_ptr(
        base=dQ + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    dK_ptr = tl.make_block_ptr(
        base=dK + offset,
        shape=(seq, head_dim),
        strides=(head_dim, 1),
        offsets=(0, 0),
        block_shape=(seq, chunk_dim),
        order=(1, 0),
    )
    for _ in range(head_chunk):
        q_data = tl.load(Q_ptr, boundary_check=(0, 1), padding_option="zero")
        k_data = tl.load(K_ptr, boundary_check=(0, 1), padding_option="zero")
        dQ_data = tl.dot(dS, k_data)
        tl.store(dQ_ptr, dQ_data.to(Q.dtype.element_ty), boundary_check=(0, 1))
        dK_data = tl.dot(dS.trans(1, 0), q_data)
        tl.store(dK_ptr, dK_data.to(K.dtype.element_ty), boundary_check=(0, 1))
        Q_ptr = tl.advance(Q_ptr, (0, chunk_dim))
        K_ptr = tl.advance(K_ptr, (0, chunk_dim))
        dQ_ptr = tl.advance(dQ_ptr, (0, chunk_dim))
        dK_ptr = tl.advance(dK_ptr, (0, chunk_dim))

# Autograd function for Flash Window Attention
class FlashWindowAttnFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, scale_qk):
        batch, heads, seq, head_dim = q.shape
        assert head_dim % 16 == 0, "head_dim must be divisible by 16"
        o = torch.empty_like(q)
        head_chunk = head_dim // 16  # r = C/16 as per paper
        grid = (batch, heads, 1)
        _flash_window_fwd_kernel[grid](
            q, k, v, o, scale_qk,
            batch, heads, seq, head_dim, head_chunk, 16
        )
        ctx.save_for_backward(q, k, v)
        ctx.scale_qk = scale_qk
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v = ctx.saved_tensors
        batch, heads, seq, head_dim = q.shape
        dq = torch.empty_like(q)
        dk = torch.empty_like(k)
        dv = torch.empty_like(v)
        head_chunk = head_dim // 16
        grid = (batch, heads, 1)
        _flash_window_bwd_kernel[grid](
            q, k, v, do, dq, dk, dv,
            ctx.scale_qk, batch, heads, seq, head_dim, head_chunk, 16
        )
        return dq, dk, dv, None

# Flash Window Multi-Head Self-Attention
# Flash Window Multi-Head Self-Attention
class FlashWindowMSA(nn.Module):
    def __init__(self, dim, heads, win, qk_dim=None):
        super().__init__()
        self.w = win
        self.heads = heads
        self.qk_dim = qk_dim if qk_dim is not None else dim

        self.qkv = nn.Linear(dim, dim * 3)
        head_dim = dim // heads
        qk_head_dim = self.qk_dim // heads  # expanded head dim when qk_dim is provided

        # project Q and K to qk_head_dim; ALSO project V to the same dim
        self.qk_proj = nn.Linear(head_dim, qk_head_dim) if qk_dim is not None else nn.Identity()
        self.v_proj  = nn.Linear(head_dim, qk_head_dim) if qk_dim is not None else nn.Identity()

        self.scale = (self.qk_dim // heads) ** -0.5  # scale matches projected QK dim

        # output projection must accept heads * qk_head_dim (not dim) when qk_dim is used
        self.proj = nn.Linear(qk_head_dim * heads, dim)

    def forward(self, x):  # x: B C H W
        
        B, C, H, W = x.shape
        w = self.w
    
        # Dynamic padding (so H, W are divisible by w)
        pad_h = (w - H % w) % w
        pad_w = (w - W % w) % w
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        H_pad, W_pad = H + pad_h, W + pad_w
    
        # Window partition → tokens (B_win, L, C)
        tokens = _wpart(x, w).flatten(2).transpose(1, 2)
        B_win, L, _ = tokens.shape
        qkv = self.qkv(tokens).reshape(B_win, L, 3, self.heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B*nh*nw) heads L head_dim

        # project all of Q, K, and V to the same head dim
        q = self.qk_proj(q)
        k = self.qk_proj(k)
        v = self.v_proj(v)

        out = FlashWindowAttnFunc.apply(q, k, v, self.scale)

        # Project and unpartition
        out = self.proj(out.transpose(1, 3).reshape(B_win, L, -1))  # (B*nh*nw) L C
        out = out.transpose(1, 2).view(-1, C, w, w)
        out = _wunpart(out, w, H_pad, W_pad, B)

        # Remove padding
        if pad_h or pad_w:
            out = out[:, :, :H, :W]
        return out

# Depth-wise Separable Convolution Block
class DWConvBN(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch), nn.GELU(),
            nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x):
        return self.net(x)


# Fractal Information-Flow Attention (L1 + L2)
class FIFMAtt(nn.Module):
    def __init__(self, dim, heads, p=4, s=2, tau=1.0):
        super().__init__()
        self.tau = tau
        self.p = p      # Local window size
        self.s = s      # Grouping factor  
        self.P = p * s  # Non-local window size
        
        # Layer norms for L1 and L2 levels
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        
        # L1: Local fractal layer (p×p windows)
        assert (dim // heads) % 16 == 0, f"head_dim ({dim // heads}) must be divisible by 16"
        self.l1 = FlashWindowMSA(dim, heads, p)
        
        # L2: Non-local fractal aggregation (P×P windows with higher-dim QK space)
        self.l2 = FlashWindowMSA(dim, heads, self.P, qk_dim=dim*2)
        
    def _ln_chw(self, x, ln):
        """Apply LayerNorm to CHW format tensor"""
        B, C, H, W = x.shape
        y = x.flatten(2).transpose(1, 2)
        y = ln(y).transpose(1, 2).view(B, C, H, W)
        return y
    
    def _fractal_reshape_l2(self, x):
        """Group s×s local patches into P×P regions for L2"""
        B, C, H, W = x.shape
        p, s, P = self.p, self.s, self.P
        
        # Dynamic padding
        pad_h = (P - H % P) % P
        pad_w = (P - W % P) % P 
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
            H_pad, W_pad = H + pad_h, W + pad_w
        else:
            H_pad, W_pad = H, W
            
        # Fractal aggregation: p×p → P×P grouping
        nh_p, nw_p = H_pad // p, W_pad // p
        x_local = x.view(B, C, nh_p, p, nw_p, p)
        
        nh_P, nw_P = nh_p // s, nw_p // s
        x_grouped = x_local.view(B, C, nh_P, s, p, nw_P, s, p)
        x_fractal = x_grouped.permute(0, 2, 5, 1, 3, 6, 4, 7)
        x_fractal = x_fractal.reshape(B * nh_P * nw_P, C, P, P)
        
        return x_fractal, (H_pad, W_pad, nh_P, nw_P, pad_h, pad_w)
    
    def _fractal_unreshape_l2(self, x, reshape_info):
        """Reverse fractal aggregation"""
        H_pad, W_pad, nh_P, nw_P, pad_h, pad_w = reshape_info
        B = x.shape[0] // (nh_P * nw_P)
        C, P = x.shape[1], x.shape[2]
        p, s = self.p, self.s
        
        x = x.view(B, nh_P, nw_P, C, s, s, p, p)
        x = x.permute(0, 3, 1, 4, 6, 2, 5, 7)
        x = x.reshape(B, C, nh_P * s * p, nw_P * s * p)
        
        if pad_h or pad_w:
            x = x[:, :, :H_pad-pad_h, :W_pad-pad_w]
            
        return x
    
    def forward(self, x):
        """L1 (local) → L2 (non-local fractal aggregation)"""
        # L1: Local Fractal Layer
        x_ln1 = self._ln_chw(x, self.ln1)
        y1 = self.l1(x_ln1)
        x = x + self.tau * y1
        
        # L2: Non-Local Fractal Aggregation Layer  
        x_fractal, reshape_info = self._fractal_reshape_l2(x)
        y2 = self.l2(x_fractal)
        y2_reshaped = self._fractal_unreshape_l2(y2, reshape_info)
        y2_ln = self._ln_chw(y2_reshaped, self.ln2)
        x = x + self.tau * y2_ln
        
        return x

# Fractal Feed-Forward Convolution (L3)
class FIFMConv(nn.Module):
    """L3 Global Modelling Layer using your existing IMLP"""
    def __init__(self, dim, mlp_ratio=2.0, tau=1.0):
        super().__init__()
        self.tau = tau
        self.ln = nn.LayerNorm(dim)
        
        # Spatial convolution for global context
        self.dw_conv = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim, bias=False),
            nn.BatchNorm2d(dim),
            nn.GELU()
        )
        
        # Use your existing IMLP
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = IMLP(dim, hidden_dim, spatial_ch=None)
        
    def _ln_chw(self, x):
        B, C, H, W = x.shape
        y = x.flatten(2).transpose(1, 2)
        y = self.ln(y).transpose(1, 2).view(B, C, H, W)
        return y
        
    def forward(self, x):
        x_ln = self._ln_chw(x)
        y = self.dw_conv(x_ln)  # Spatial processing
        y = self.mlp(y)         # Your IMLP for channel processing
        return x + self.tau * y
# Fractal Block
class FracBlock(nn.Module):
    def __init__(self, dim, heads, p, s, tau=1.0, mlp_ratio=2.0):
        super().__init__()
        self.att = FIFMAtt(dim, heads, p, s, tau)
        self.conv = FIFMConv(dim, mlp_ratio=mlp_ratio, tau=tau)
    def forward(self, x):
        x = self.att(x)
        x = self.conv(x)
        return x

# Fractal Classification Model
class FractalCLS(nn.Module):
    CFG = {
        'T': dict(ed=64, depths=[2, 2, 6, 2], heads=[2, 4, 8, 16]),
        'S': dict(ed=48, depths=[1, 2, 4, 2], heads=[3, 6, 12, 24]),
        'B': dict(ed=128, depths=[2, 2, 18, 2], heads=[4, 8, 16, 32]),
    }

    def __init__(self, n_cls=10, size='T', p=4, s=2, tau=1.0):
        super().__init__()
        cfg = self.CFG[size]
        self.stem = nn.Sequential(
            DWConvBN(3, cfg['ed'], stride=p),
            nn.BatchNorm2d(cfg['ed']), nn.GELU())
        dims = [cfg['ed'], cfg['ed']*2, cfg['ed']*4, cfg['ed']*8]
        wins = [p, p, p, p]  # Fix: Use p=4 for all stages to avoid dimension reduction
        stages = []
        for i in range(4):
            stages += [FracBlock(dims[i], cfg['heads'][i], wins[i], s, tau)
                       for _ in range(cfg['depths'][i])]
            if i < 3:
                stages.append(nn.Sequential(
                    DWConvBN(dims[i], dims[i+1], stride=2),
                    nn.BatchNorm2d(dims[i+1]), nn.GELU()))
        self.net = nn.Sequential(*stages)
        self.norm = nn.BatchNorm2d(dims[-1])
        self.head = nn.Linear(dims[-1], n_cls)

    def forward(self, x):
        x = self.stem(x)
        x = self.net(x)
        x = self.norm(x)
        return self.head(torch.mean(x, dim=[2, 3]))
        
# ------------------------------------------------------------
# 1.  put ALL your model classes above this cell / in scope
#     (e.g. HybridResNetCNN, FractalCLS, etc.)
# 2.  list the ones you want to compare in `models_to_check`
# ------------------------------------------------------------

def count_params(model: torch.nn.Module, trainable_only: bool = True) -> int:
    """Return parameter count (optionally only those with gradients)."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())

# ── instantiate the models you want to inspect ──────────────
models_to_check = {
    # "ResNet18-FT": models.resnet18(weights=None, num_classes=10),     # example
    # "HybridResNetCNN": HybridResNetCNN(num_classes=10),
    "FractalCLS-T": FractalCLS(n_cls=10, size="S"),                  
}

# ── loop & print ─────────────────────────────────────────────
for name, net in models_to_check.items():
    n_params = count_params(net)
    print(f"{name:<20s}: {n_params/1e6:6.2f} M parameters")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Define the paths to your dataset CSV files
ROOT = "/kaggle/input/eurosat-dataset/EuroSAT"
CSV_T  = f"{ROOT}/train.csv"      # Path to the training CSV file
CSV_V  = f"{ROOT}/validation.csv" # Path to the validation CSV file
CSV_TE = f"{ROOT}/test.csv"       # Path to the test CSV file

# ---------- hyper-params ---------------------------------------------------
BATCH_SIZE   = 32
EPOCHS_FRAC  = 2          # tweak as needed
NUM_CLASSES  = 10
LEARNING_RATE= 5e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"▶ Device: {device}")

tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

# ---------- data loaders ---------------------------------------------------
train_loader = DataLoader(EuroSATDataset(CSV_T, ROOT, tfm), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(EuroSATDataset(CSV_V, ROOT, tfm), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(EuroSATDataset(CSV_TE, ROOT, tfm), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

# ---------------- Grad-CAM Class ----------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate_cam(self, input_tensor, class_idx=None):
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        weighted_activations = self.activations[0] * pooled_gradients[:, None, None]
        cam = weighted_activations.sum(dim=0)

        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

# ---------------- Load Model and Apply Grad-CAM ----------------

# Reload model and load weights
model = FractalCLS(n_cls=10, size='S').to(device)
model.load_state_dict(torch.load("fractalcls_s_eurosat.pth"))
model.eval()

# Set the target layer for Grad-CAM
target_layer = model.net[-3]  # You can experiment with -4, -5 as well
gradcam = GradCAM(model, target_layer)

# Pick an image from test dataset
image_tensor, label = next(iter(test_loader))
image_tensor = image_tensor.to(device)
image = image_tensor[0].unsqueeze(0)  # Single image batch [1, 3, 224, 224]

# Generate CAM
cam = gradcam.generate_cam(image)

# Unnormalize and convert image to numpy
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
img_np = (image[0].cpu() * std.cpu() + mean.cpu()).permute(1, 2, 0).numpy()
img_np = np.clip(img_np, 0, 1)

# Resize CAM to image size
cam_resized = cv2.resize(cam, (224, 224))
heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
heatmap = np.float32(heatmap) / 255
overlay = heatmap + img_np
overlay = overlay / overlay.max()

# Visualize
plt.figure(figsize=(10, 5))
plt.imshow(overlay)
plt.title("Grad-CAM: Top-1 Predicted Class")
plt.axis("off")
plt.show()

# Clean up hooks
gradcam.remove_hooks()


In [ ]:
import os
import re
from pathlib import Path

# --- create output folder ---
out_dir = Path("top1_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

NUM_IMAGES = 5  # Number of test images to visualize
model.eval()

# Grad-CAM setup
target_layer = model.net[-3]
gradcam = GradCAM(model, target_layer)

# Normalization helpers
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

count = 0
for image_tensor_batch, label_batch in test_loader:
    image_tensor_batch = image_tensor_batch.to(device)

    for i in range(image_tensor_batch.size(0)):
        image = image_tensor_batch[i].unsqueeze(0)  # [1, 3, 224, 224]
        cam = gradcam.generate_cam(image)           # 2D CAM, typically [H, W] in 0..1

        # Optional: get top-1 prediction for naming
        with torch.no_grad():
            logits = model(image)
            probs = torch.softmax(logits, dim=1)
            top1_prob, top1_idx = probs.max(dim=1)
        pred_name = (
            class_names[top1_idx.item()]
            if "class_names" in globals() and len(class_names) > top1_idx.item()
            else f"class_{top1_idx.item()}"
        )

        safe_class = re.sub(r'[^A-Za-z0-9]+', '_', pred_name).strip('_')
        
        # Unnormalize original image -> RGB float 0..1
        img_np = (image[0].cpu() * std.cpu() + mean.cpu()).permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)

        # Resize CAM and make heatmap
        cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
        heatmap_bgr = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)  # uint8 BGR
        heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)                       # uint8 RGB
        heatmap_rgb_f = heatmap_rgb.astype(np.float32) / 255.0

        # Overlay (weighted blend in RGB)
        overlay = np.clip(0.6 * heatmap_rgb_f + 0.4 * img_np, 0.0, 1.0)

        # ---- SHOW side-by-side ----
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(img_np)
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(overlay)
        plt.title(f"Grad-CAM: {pred_name} ({top1_prob.item():.2f})")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        # ---- SAVE files ----
        base = out_dir / f"Top-1 Prediction_{safe_class}"

        # 1) Original image (PNG)
        orig_rgb_uint8 = (img_np * 255).astype(np.uint8)
        cv2.imwrite(str(base.with_suffix(".orig.png")),
                    cv2.cvtColor(orig_rgb_uint8, cv2.COLOR_RGB2BGR))

        # 2) Heatmap only (PNG)
        cv2.imwrite(str(base.with_suffix(".heatmap.png")), heatmap_bgr)

        # 3) Overlay only (PNG)
        overlay_rgb_uint8 = (overlay * 255).astype(np.uint8)
        cv2.imwrite(str(base.with_suffix(".overlay.png")),
                    cv2.cvtColor(overlay_rgb_uint8, cv2.COLOR_RGB2BGR))

        # 4) Side-by-side figure (PNG)
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(orig_rgb_uint8)
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(overlay_rgb_uint8)
        plt.title(f"Grad-CAM: {pred_name} ({top1_prob.item():.2f})")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(str(base.with_suffix(".figure.png")), dpi=150, bbox_inches="tight")
        plt.close()

        count += 1
        if count >= NUM_IMAGES:
            break
    if count >= NUM_IMAGES:
        break

# Remove hooks
gradcam.remove_hooks()


In [ ]:
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
os.makedirs("grad_cam_outputs", exist_ok=True)

# ---------------- Grad-CAM Class ----------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate_cam(self, input_tensor, class_idx=None):
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        weighted_activations = self.activations[0] * pooled_gradients[:, None, None]
        cam = weighted_activations.sum(dim=0)

        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), output.argmax(dim=1).item()

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

# ---------------- Setup ----------------

# Load label_map to convert class index to class name
with open("/kaggle/input/eurosat-dataset/EuroSAT/label_map.json", 'r') as f:
    label_map = json.load(f)
idx_to_label = {v: k for k, v in label_map.items()}

# Reload trained model
model = FractalCLS(n_cls=10, size='S').to(device)
model.load_state_dict(torch.load("fractalcls_s_eurosat.pth"))
model.eval()

# Select the Grad-CAM target layer
target_layer = model.net[-3]
gradcam = GradCAM(model, target_layer)

# Normalization helpers
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(device)

# Visualize 5 images
NUM_IMAGES = 15
count = 0

for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

    for i in range(images.size(0)):
        image = images[i].unsqueeze(0)  # [1, 3, 224, 224]
        true_class = labels[i].item()

        # Get CAM and predicted class
        cam, pred_class = gradcam.generate_cam(image)

        # Unnormalize image
        img_np = (image[0].cpu() * std.cpu() + mean.cpu()).permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)

        # Resize and overlay CAM
        cam_resized = cv2.resize(cam, (224, 224))
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        overlay = heatmap + img_np
        overlay = overlay / overlay.max()

        # ---- Plot all 4 panels ----


        fig = plt.figure(figsize=(10, 6))   # assign to fig

        plt.subplot(2, 2, 1)
        plt.imshow(img_np)
        plt.title(f"Original Image\nTrue: {idx_to_label[true_class]}")
        plt.axis("off")

        plt.subplot(2, 2, 2)
        plt.imshow(overlay)
        plt.title("Grad-CAM on Original")
        plt.axis("off")

        plt.subplot(2, 2, 3)
        plt.imshow(img_np)
        plt.title(f"Predicted: {idx_to_label[pred_class]}")
        plt.axis("off")

        plt.subplot(2, 2, 4)
        plt.imshow(overlay)
        plt.title("Grad-CAM on Prediction")
        plt.axis("off")

        plt.suptitle(f"Test Sample {count+1}: {idx_to_label[true_class]} → {idx_to_label[pred_class]}", fontsize=14)
        plt.tight_layout()
        save_path = os.path.join("grad_cam_outputs", f"gradcam_{count+1}_{idx_to_label[true_class]}_to_{idx_to_label[pred_class]}.png")
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved Grad-CAM visualization to {save_path}")
        plt.show()

        count += 1
        if count >= NUM_IMAGES:
            break
    if count >= NUM_IMAGES:
        break

gradcam.remove_hooks()
